# Day 13 Capstone - Completed Submission

## PHASE 0 (Confirmed)
- **Domain:** E-Commerce Customer Support Assistant
- **User:** Online shoppers who ask about returns, refunds, shipping, tracking, and order issues
- **Problem Statement:** E-commerce support teams receive repetitive policy and order queries, increasing response time and workload. A grounded agent is needed to answer policy questions consistently, use tools for date/calculation requests, and preserve context across turns.
- **Success Criteria:** retrieval quality gate passes, faithfulness threshold is enforced with retry loop, memory works via thread_id, tool routing works, and out-of-scope/injection prompts are handled safely without hallucination.

In [ ]:
from agent import build_agent, CapstoneState, DOCUMENTS

assistant = build_agent(enforce_retrieval_gate=True, verbose=True)

## PHASE 1 - Knowledge Base + Retrieval Gate

In [ ]:
print(f"Knowledge base documents: {len(DOCUMENTS)}")
for doc in DOCUMENTS:
    print(f"- {doc['id']} | {doc['topic']} | words={len(doc['text'].split())}")

print(f"\nRetrieval gate score: {assistant.retrieval_score:.2f}")
for item in assistant.retrieval_report:
    print({"query": item.query, "top_topics": item.topics, "pass": item.passed})

## PHASE 2 - State Design

In [ ]:
print("CapstoneState fields:")
print(list(CapstoneState.__annotations__.keys()))

## PHASE 3 - Node Functions (Isolated Tests)

In [ ]:
assistant.run_node_tests()
print("All node tests passed.")

## PHASE 4 - Graph Assembly

In [ ]:
print("Graph compiled successfully")
print(f"LLM backend: {assistant.llm_backend}")
print(assistant.app)

## PHASE 5 - Testing (10+ including Red-Team) with PASS/FAIL

In [ ]:
phase5_results = assistant.run_phase5_tests()

for row in phase5_results:
    print("-" * 90)
    print(f"Q: {row['question']}")
    print(f"route={row['route']} | faithfulness={row['faithfulness']:.2f} | red_team={row['red_team']}")
    print(f"status={'PASS' if row['passed'] else 'FAIL'}")
    print(f"answer={row['answer'][:260]}")

phase5_passed = sum(1 for row in phase5_results if row['passed'])
print(f"\nPhase 5 summary: {phase5_passed}/{len(phase5_results)} passed")

In [ ]:
memory_check = assistant.run_memory_sequence_test()
print(memory_check)

## PHASE 6 - Evaluation (RAGAS or LLM fallback)

In [ ]:
phase6_report = assistant.run_phase6_evaluation()
print(phase6_report)

## PHASE 7 - Streamlit Deployment

Run from terminal:

```bash
streamlit run capstone_streamlit.py
```

## PHASE 8 - Written Summary

**Name:** Aastha  
**Domain chosen:** E-Commerce Customer Support Assistant  
**What the agent does:** Answers customer support questions about returns, refunds, shipping, cancellation, damaged/wrong items, and escalation rules using grounded retrieval and controlled routing. It uses thread-based memory to preserve context in multi-turn conversations and supports tool routes for date/math questions.  
**Knowledge base:** 10 structured policy documents, one topic per document, embedded with `all-MiniLM-L6-v2`, indexed in ChromaDB in-memory collection.  
**Tool used:** Datetime + calculator tool for non-KB operational queries.  
**RAGAS baseline path:** `phase6_report` from the evaluation cell above (RAGAS if installed, otherwise fallback scoring).  
**Test results:** `phase5_passed / len(phase5_results)` from Phase 5. Red-team tests included in the same run.  
**One thing to improve with more time:** Add hybrid retrieval (BM25 + vector re-ranking) and source citation spans for each sentence in the answer.  
**Most surprising thing learned:** Retrieval quality and strict state design affect reliability more than prompt wording alone.

In [ ]:
warnings_status = assistant.helper_warnings_status()
print("Final checklist")
print({
    "no_todo_left": True,
    "kb_docs": len(DOCUMENTS),
    "retrieval_score": assistant.retrieval_score,
    "phase5_passed": phase5_passed,
    "phase5_total": len(phase5_results),
    "memory_sequence_passed": memory_check["passed"],
    "phase6_report": phase6_report,
    "warning_compliance": warnings_status,
})